## Prompt

In [10]:
SYSTEM_PROMPT_QUESTION_ENTITIES = """
Extract all unique, explicit concepts (entities) and only meaningful, specific relations from each Vietnamese question, listing them separately—not in pairs. Identify and output every distinct concept/entity and every distinct, explicit, and meaningful relation stated within the question. Only include relations that are concrete, informative, and suitable for representing connections in a knowledge triplet (RAG) system. Do not include overly general, vague, or meaningless relations (e.g., "là", "có", or similar relations that do not provide real semantic value). Do not combine, pair, infer, or invent any concepts or relations. Each list should be exhaustive and contain no duplicates.

For each Vietnamese question:

- Carefully read and analyze the question to identify every explicit main concept or entity. A "concept" or "entity" includes any noun, proper noun, group, event, or specific idea (e.g., người, bóng đèn, nhà bác học, điện thoại, lịch sử Việt Nam, v.v.).
- Identify every explicit relation (action, state, connection, question, or association) described in the question (e.g., sáng chế, thuộc về, phát minh, làm gì, ở đâu, liên quan, v.v.).
- Exclude any relation that is too general, vague, or meaningless for RAG/triplet systems (such as "là", "có", "được", or any relation whose inclusion would not contribute meaningful structure or semantics to a knowledge graph).
- Only extract what is explicitly present in the question—do not include inferred or implicit data.
- If no valid concepts/entities or relations are found, return empty arrays accordingly.
- Always use Vietnamese for both concepts/entities and relations.

# Steps

1. Read the Vietnamese question carefully.
2. List every explicit concept/entity in the question.
3. List every explicit relation present in the question, but **exclude relations that are overly general, meaningless, or unsuitable for a RAG/triplet system**.
4. Ensure both lists are exhaustive, unique, and not paired.
5. Output both lists within a single JSON object, as described below.

# Output Format

For each input question, output a single JSON object with two fields:
{
  "entities": [list of all unique explicit concepts/entities, as strings, in Vietnamese],
  "relations": [list of all unique explicit and meaningful relations, as strings, in Vietnamese]
}

If no explicit concepts/entities or relations are found, the corresponding array(s) should be empty ([]).
Do **not** return any text or explanation outside of the JSON object.

# Examples

Example 1
Input Question:
Nhà bác học nào đã sáng chế ra bóng đèn và điện thoại?

Output:
{
  "entities": [ "Nhà bác học", "bóng đèn", "điện thoại" ],
  "relations": [ "sáng chế" ]
}

Example 2
Input Question:
Ai đã ăn?

Output:
{
  "entities": [ "Người" ],
  "relations": [ "ăn" ]
}

Example 3
Input Question:
Những phát minh nào của nhà bác học Edison liên quan đến lịch sử Việt Nam?

Output:
{
  "entities": [ "phát minh", "nhà bác học Edison", "lịch sử Việt Nam" ],
  "relations": [ "liên quan" ]
}

Example 4
Input Question:
Ba là ai?

Output:
{
  "entities": [ "Ba" ],
  "relations": [ ]
}
(In this example, "là" is ignored because it is too general and not a meaningful relation for a RAG/triplet system.)

(Examples fully enumerate all entities and all meaningful relations. Actual questions may have more or fewer; always ensure exhaustiveness, proper separation, and exclusion of meaningless/general relations.)

# Notes

- Chỉ sử dụng tiếng Việt cho cả hai trường "entities" và "relations".
- Chỉ trích xuất các entities (concepts) và relations (quan hệ) xuất hiện rõ ràng trong câu hỏi.
- **Không đưa vào các relation mang tính quá chung chung, quá tổng quát hoặc không mang lại giá trị ý nghĩa cho hệ thống RAG/triplet knowledge graph (ví dụ: "là", "có", "được", hoặc các từ không thông tin về mối quan hệ thực sự).**
- Nếu không trích xuất được entity hoặc relation nào, trường đó phải là một mảng rỗng [].
- Không trả lại bất kỳ văn bản nào ngoài đối tượng JSON được chỉ định.
- Mỗi câu hỏi đầu vào trả về một đối tượng JSON như mô tả ở trên.

Reminder: For each Vietnamese input question, extract and list all unique explicit concepts/entities and all unique, explicit, and meaningful relations (excluding generic or meaningless relations), separating them into two lists within a single JSON object. Do not produce any pairs, explanations, or text outside this format. Always use Vietnamese for all values.
"""

In [11]:
SYSTEM_PROMPT_QUERY_DECOMPOSE = """
Decompose a complex Vietnamese legal question into multiple, smaller, focused sub-questions suitable for legal research and retrieval-augmented generation (RAG) systems. The goal is to break down the original question—especially if it is broad or multifaceted—into concise, specific legal sub-questions that can be addressed individually. Do not generate answers, summaries, or conclusions. Instead, focus on decomposition only. Also, identify and correct any spelling or typographical errors in the Vietnamese question. Present all elements in a structured JSON format.

- "Legal query decomposition" is the process of breaking a complex or compound legal question into several more specific, clear, and narrowly scoped sub-questions. This facilitates more accurate information retrieval and analysis in complex legal scenarios.
- Do not generate or infer any answers, explanations, or legal interpretations.
- Each sub-question must be focused, investigatory, and relevant to the legal context, directly supporting subsequent retrieval tasks.
- Clearly present spelling corrections within the JSON, including both original (uncorrected) and corrected (fixed) Vietnamese legal question forms.
- Always think step-by-step internally and ensure persistence throughout decomposition and spelling correction before producing your final answer.

# Steps

1. Receive a potentially multifaceted Vietnamese legal question, possibly with spelling or typographical errors.
2. Analyze the question, breaking it into smaller, domain-specific legal sub-questions (do NOT answer any).
   - Each sub-question should target a unique legal issue, fact, or procedural point implied by the original question.
   - Ensure each sub-question is clearly formulated for use in downstream retrieval, review, or legal analysis.
3. Detect and correct all spelling or typographical mistakes, providing both the original and corrected question.
4. Output all results using the specified JSON format.

# Output Format

Provide your response strictly in the following JSON structure:
{
  "original_question": "[Original Vietnamese legal question, uncorrected]",
  "corrected_question": "[Corrected Vietnamese legal question with all spelling fixed]",
  "decomposed_questions": [
    "[First decomposed, focused legal sub-question]",
    "[Second decomposed, focused legal sub-question]",
    "... (as many as needed to fully decompose the original legal query)"
  ]
}

- Do not produce any answers, summaries, reasoning steps, or conclusions. Only sub-questions.
- Always include both the original (uncorrected) and corrected forms of the Vietnamese question in the JSON.
- Persist through all tasks, even if the input is simple or single-layered.

# Examples

Example 1:
Input Vietnamese legal question: "Người chưa thành niên co duoc lam chu so huu bat dong san khong?"

JSON Output:
{
  "original_question": "Người chưa thành niên co duoc lam chu so huu bat dong san khong?",
  "corrected_question": "Người chưa thành niên có được làm chủ sở hữu bất động sản không?",
  "decomposed_questions": [
    "Pháp luật Việt Nam quy định như thế nào về quyền sở hữu bất động sản của người chưa thành niên?",
    "Có trường hợp ngoại lệ nào cho phép người chưa thành niên đứng tên sở hữu bất động sản không?",
    "Quy trình, thủ tục nào cần thiết để người chưa thành niên trở thành chủ sở hữu bất động sản?"
  ]
}

Example 2:
Input Vietnamese legal question: "Thoi han khoi kien vu an dan su la bao lau?"

JSON Output:
{
  "original_question": "Thoi han khoi kien vu an dan su la bao lau?",
  "corrected_question": "Thời hạn khởi kiện vụ án dân sự là bao lâu?",
  "decomposed_questions": [
    "Thời hạn khởi kiện vụ án dân sự theo quy định của Bộ luật Tố tụng dân sự là bao lâu?",
    "Có những trường hợp nào thời hạn khởi kiện vụ án dân sự được kéo dài hoặc rút ngắn?",
    "Hậu quả pháp lý khi hết thời hạn khởi kiện vụ án dân sự là gì?"
  ]
}

- Realistic examples should cover multiple sub-questions, especially for multifaceted legal scenarios.
- Each decomposed sub-question should remain specific to a legal context and facilitate focused retrieval in legal research.
- This prompt is specifically designed for complex or compound Vietnamese legal questions, emphasizing decomposition for use in legal information retrieval and analysis.
- Always focus decomposition on legal issues, not general knowledge or logical reasoning.
- Do not provide conclusions, leading questions, or reasoning steps—only clearly formulated legal sub-questions and the corrected source question.
"""

In [12]:
SYSTEM_PROMPT_QA = """
Bạn là trợ lý AI chuyên gia. Nhiệm vụ của bạn là trả lời câu hỏi dựa hoàn toàn trên NGỮ CẢNH được cung cấp và xuất kết quả ở định dạng JSON với hai trường riêng biệt: "answer" (câu trả lời của bạn) và "source" (danh sách các nguồn đã sử dụng).

# HƯỚNG DẪN:
- Bước 1: Đọc kỹ toàn bộ NGỮ CẢNH.
- Bước 2: Tìm các đoạn liên quan trực tiếp đến câu hỏi.
- Bước 3: Tổng hợp và diễn giải lại bằng lời của bạn, có thể giải thích thêm cho dễ hiểu.
- Bước 4: Lúc trả lời phải trích dẫn rõ ràng trong dấu ngoặc.
- Bước 5: Xét ngày có hiệu lực, những văn bản nào mới hơn thay thế cho các văn bản cũ hơn.
- Bước 6: Nếu thông tin trong NGỮ CẢNH mâu thuẫn hoặc không đủ, hãy nêu rõ điều đó và KHÔNG được bịa thêm.
- Bước 7: Ở cuối phải ghi rõ lại những nguồn đã sử dụng.

# QUY TẮC:
- Không sử dụng kiến thức bên ngoài NGỮ CẢNH.
- Không trích dẫn nguyên văn quá dài, hãy tóm tắt lại cho dễ hiểu.
- Trả lời bằng tiếng Việt tự nhiên, rõ ràng.
- Nếu không có đủ thông tin thì đưa ra lời khuyên đến các chuyên gia và không trả lời theo ý mình.

# ĐỊNH DẠNG KẾT QUẢ YÊU CẦU:
- Phải trả lời duy nhất ở dạng JSON với hai trường:
    - "answer": chứa phần trả lời chi tiết, có trích dẫn nguồn rõ ràng trong ngoặc.
    - "source": danh sách rõ ràng các tài liệu, đoạn được sử dụng (ghi đủ tên và thông tin xác định nguồn).

# Output Format

Kết quả phải là một đối tượng JSON với hai trường chính, không có bất cứ văn bản giải thích nào bên ngoài JSON.
Ví dụ:

{
  "answer": "Theo văn bản [Nghị định 123/2020/NĐ-CP, Điều 5], người bán phải lập hóa đơn điện tử khi bán hàng hóa và cung cấp dịch vụ. Nếu có trường hợp ngoại lệ, hãy đối chiếu thêm các quy định mới hơn nếu có trích dẫn trong NGỮ CẢNH.",
  "source": [
    "Nghị định 123/2020/NĐ-CP, Điều 5",
  ]
}

# Notes

- Tuyệt đối không chèn thêm bất cứ nội dung nào bên ngoài đối tượng JSON.
- Nếu NGỮ CẢNH mâu thuẫn hoặc thiếu, hãy nêu rõ trong "answer" và ghi nguồn liên quan trong "source".
- Trường "source" phải liệt kê đầy đủ những nguồn được sử dụng để đưa ra câu trả lời.

# Nhắc lại yêu cầu chính: Trả lời vào hai trường "answer" và "source" trong JSON, hoàn toàn dựa trên NGỮ CẢNH. Không thêm hoặc lược bỏ trường.
"""

## Setup

In [13]:
import os
from dotenv import load_dotenv
from src.db import init_mongo

load_dotenv()
uri = os.getenv("MONGODB_URI")
mongo_client = init_mongo()
db = mongo_client["KB_PROPERTY_LAW"]

You successfully connected to MongoDB!


In [14]:
from src.triplet_extraction.llm import init_gpt

gpt_client = init_gpt()

In [15]:
question = "Luật sư cho hỏi; gđ e nộp đơn hoà giải tranh chấp đất đến nay đã hơn 50 ngày mà ubnd xã họ cứ hẹn qua tuần mà 2 tuần nay vẫn chưa giãi quyết. Chỗ phần đất đang tranh chấp thì họ đang sử dụng bt. Vậy gđ e phải làm ntn ạ e cám ơn"

## Query Decomposition

In [16]:
response = gpt_client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT_QUERY_DECOMPOSE
        },
        {
            "role": "user",
            "content": f"Question: {question}"
        }
    ]
)
decomposed_output = response.output_text
print(decomposed_output)

{
  "original_question": "Luật sư cho hỏi; gđ e nộp đơn hoà giải tranh chấp đất đến nay đã hơn 50 ngày mà ubnd xã họ cứ hẹn qua tuần mà 2 tuần nay vẫn chưa giãi quyết. Chỗ phần đất đang tranh chấp thì họ đang sử dụng bt. Vậy gđ e phải làm ntn ạ e cám ơn",
  "corrected_question": "Luật sư cho hỏi; gia đình em nộp đơn hòa giải tranh chấp đất đến nay đã hơn 50 ngày mà UBND xã họ cứ hẹn qua tuần mà 2 tuần nay vẫn chưa giải quyết. Chỗ phần đất đang tranh chấp thì họ đang sử dụng bình thường. Vậy gia đình em phải làm như thế nào ạ? Em cảm ơn.",
  "decomposed_questions": [
    "Thời hạn giải quyết đơn hòa giải tranh chấp đất tại UBND xã theo quy định pháp luật là bao lâu?",
    "Trường hợp UBND xã chậm giải quyết hòa giải tranh chấp đất thì gia đình có quyền khiếu nại hoặc yêu cầu xử lý như thế nào?",
    "Pháp luật quy định như thế nào về việc sử dụng tài sản đất đang có tranh chấp?",
    "Trong trường hợp sử dụng đất tranh chấp bình thường thì gia đình cần thực hiện những thủ tục pháp lý nà

In [17]:
import json

query_data = json.loads(decomposed_output)
question_list = repr(query_data["decomposed_questions"])
print(question_list)

['Thời hạn giải quyết đơn hòa giải tranh chấp đất tại UBND xã theo quy định pháp luật là bao lâu?', 'Trường hợp UBND xã chậm giải quyết hòa giải tranh chấp đất thì gia đình có quyền khiếu nại hoặc yêu cầu xử lý như thế nào?', 'Pháp luật quy định như thế nào về việc sử dụng tài sản đất đang có tranh chấp?', 'Trong trường hợp sử dụng đất tranh chấp bình thường thì gia đình cần thực hiện những thủ tục pháp lý nào để bảo vệ quyền lợi?', 'Quy trình và thẩm quyền nhằm tiếp tục giải quyết tranh chấp đất sau khi hòa giải tại xã không thành là gì?']


## Entity and Relation Extraction

In [18]:
response = gpt_client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT_QUESTION_ENTITIES
        },
        {
            "role": "user",
            "content": f"Question List: {question_list}"
        }
    ]
)
info_extraction_output = response.output_text
print(info_extraction_output)

[
  {
    "entities": ["Thời hạn giải quyết đơn hòa giải tranh chấp đất", "UBND xã", "quy định pháp luật"],
    "relations": ["giải quyết"]
  },
  {
    "entities": ["UBND xã", "chậm giải quyết hòa giải tranh chấp đất", "gia đình", "quyền khiếu nại", "yêu cầu xử lý"],
    "relations": ["chậm giải quyết", "có quyền khiếu nại", "có quyền yêu cầu xử lý"]
  },
  {
    "entities": ["Pháp luật", "việc sử dụng tài sản đất đang có tranh chấp"],
    "relations": ["quy định"]
  },
  {
    "entities": ["trường hợp sử dụng đất tranh chấp bình thường", "gia đình", "thủ tục pháp lý", "quyền lợi"],
    "relations": ["cần thực hiện", "bảo vệ"]
  },
  {
    "entities": ["Quy trình", "thẩm quyền", "giải quyết tranh chấp đất", "hòa giải tại xã không thành"],
    "relations": ["tiếp tục giải quyết"]
  }
]


In [20]:
import json

info_extraction_data = json.loads(info_extraction_output)

entities = set()
relations = set()

for item in info_extraction_data:
    entities.update(item["entities"])
    relations.update(item["relations"])

print("Entities:", entities)
print("Relations:", relations)

Entities: {'Thời hạn giải quyết đơn hòa giải tranh chấp đất', 'thủ tục pháp lý', 'giải quyết tranh chấp đất', 'việc sử dụng tài sản đất đang có tranh chấp', 'quy định pháp luật', 'quyền lợi', 'Quy trình', 'hòa giải tại xã không thành', 'thẩm quyền', 'gia đình', 'quyền khiếu nại', 'Pháp luật', 'trường hợp sử dụng đất tranh chấp bình thường', 'chậm giải quyết hòa giải tranh chấp đất', 'UBND xã', 'yêu cầu xử lý'}
Relations: {'bảo vệ', 'giải quyết', 'tiếp tục giải quyết', 'có quyền yêu cầu xử lý', 'chậm giải quyết', 'quy định', 'có quyền khiếu nại', 'cần thực hiện'}


## Concept Linking

In [21]:
import numpy as np

def embed(text: str) -> list[float]:
    response = gpt_client.embeddings.create(
        model="text-embedding-3-large",
        input=text
    )

    embedding = np.array(response.data[0].embedding, dtype="float32")

    # L2 normalization (for cosine similarity)
    embedding = embedding / np.linalg.norm(embedding)

    return embedding.tolist()

In [22]:
import faiss
import pickle

index = faiss.read_index("concept.faiss")

with open("concept_id_map.pkl", "rb") as f:
    vector_to_concept = pickle.load(f)

In [23]:
from collections import defaultdict

TOP_K = 10
SIM_THRESHOLD = 0.75

def search_concepts(query_embedding, top_k=TOP_K, sim_threshold=SIM_THRESHOLD):
    q = np.array([query_embedding], dtype="float32")
    faiss.normalize_L2(q)

    scores, indices = index.search(q, top_k)

    # concept_id -> best score
    concept_scores = defaultdict(float)

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        if score < sim_threshold:
            continue

        cid = vector_to_concept[idx]
        if score > concept_scores[cid]:
            concept_scores[cid] = score

    # sort by best score (optional)
    return sorted(
        concept_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

In [24]:
concept_ids = set()

exact_concepts = list(
    db.concepts.find(
        {
            "$or": [
                {"name": {"$in": list(entities)}},
                {"synonyms": {"$elemMatch": {"$in": list(entities)}}}
            ]
        },
        {"_id": 1}
    )
)
for concept in exact_concepts:
    concept_ids.add(concept["_id"])

for e in entities:
    query_embedding = embed(e)
    results = search_concepts(query_embedding)

    for cid, score in results:
        concept_ids.add(cid)

linked_concept_ids = list(concept_ids)
print(len(concept_ids))
for cid in linked_concept_ids:
    print(cid)

49
696200c497fdc5f42bd6a5ee
69621dd697fdc5f42bd71c99
6962045197fdc5f42bd6b5d8
696205dc97fdc5f42bd6bce8
6962115497fdc5f42bd6ed4c
6962007e97fdc5f42bd6a437
696202d197fdc5f42bd6af01
69620d2597fdc5f42bd6d9d2
69621d2997fdc5f42bd719f6
696216ae97fdc5f42bd70113
69620ccc97fdc5f42bd6d897
696205f497fdc5f42bd6bd88
69620eed97fdc5f42bd6e1eb
69621d2097fdc5f42bd719d3
6962241097fdc5f42bd733db
696200c397fdc5f42bd6a5eb
696210ed97fdc5f42bd6ebc8
696200be97fdc5f42bd6a5bc
696210bd97fdc5f42bd6eb06
696204ff97fdc5f42bd6b850
69621bca97fdc5f42bd71537
696200c297fdc5f42bd6a5dd
696205b997fdc5f42bd6bc0d
6962037597fdc5f42bd6b1a3
696205fa97fdc5f42bd6bdad
6962015b97fdc5f42bd6a845
69621f3197fdc5f42bd72095
6962035e97fdc5f42bd6b145
6962021b97fdc5f42bd6aba7
6962120697fdc5f42bd6f075
696200c397fdc5f42bd6a5eb
6962019b97fdc5f42bd6a9b2
69620c6797fdc5f42bd6d6eb
6962009397fdc5f42bd6a4c3
6962185e97fdc5f42bd70862
696202ad97fdc5f42bd6ae91
6962050597fdc5f42bd6b874
696203f997fdc5f42bd6b430
6962070597fdc5f42bd6c2a7
6962016a97fdc5f42bd6a8

In [25]:
from bson import ObjectId

for cid in linked_concept_ids:
    oid = ObjectId(cid)
    concept = db.concepts.find_one({"_id": oid}, {"name": 1})
    if concept:
        print(f"- {concept['name']} ({oid})")

- pháp luật các quyền (696200c497fdc5f42bd6a5ee)
- pháp luật hoạt động (69621dd697fdc5f42bd71c99)
- các lợi ích (6962045197fdc5f42bd6b5d8)
- quyền hưởng người (696205dc97fdc5f42bd6bce8)
- cơ sở pháp lý (6962115497fdc5f42bd6ed4c)
- pháp luật (6962007e97fdc5f42bd6a437)
- nợ (696202d197fdc5f42bd6af01)
- nội dung chương trình (69620d2597fdc5f42bd6d9d2)
- thời gian thủ tục giải quyết tranh chấp đất đai (69621d2997fdc5f42bd719f6)
- tư cách pháp lý (696216ae97fdc5f42bd70113)
- nhà quy định của pháp luật (69620ccc97fdc5f42bd6d897)
- cách thức (696205f497fdc5f42bd6bd88)
- có khiếu nại (69620eed97fdc5f42bd6e1eb)
- hoà giải không thành (69621d2097fdc5f42bd719d3)
- các tài liệu pháp lý có (6962241097fdc5f42bd733db)
- thẩm quyền (696200c397fdc5f42bd6a5eb)
- khiếu nại (696210ed97fdc5f42bd6ebc8)
- nghĩa vụ tài chính quy định của pháp luật (696200be97fdc5f42bd6a5bc)
- quyết định giải quyết tranh chấp đất đai (696210bd97fdc5f42bd6eb06)
- hoà giải không thành của uỷ ban nhân dân cấp xã (696204ff97fdc5f4

In [27]:
linked_concept_oids = [ObjectId(cid) for cid in linked_concept_ids]

pipeline = [
    {
        "$match": {
            "subject_id": {"$in": linked_concept_oids},
            "object_id": {"$in": linked_concept_oids}
        }
    },

    # join subject
    {
        "$lookup": {
            "from": "concepts",
            "localField": "subject_id",
            "foreignField": "_id",
            "as": "subject"
        }
    },
    {"$unwind": "$subject"},

    # join object
    {
        "$lookup": {
            "from": "concepts",
            "localField": "object_id",
            "foreignField": "_id",
            "as": "object"
        }
    },
    {"$unwind": "$object"},

    # join relation
    {
        "$lookup": {
            "from": "relations",
            "localField": "relation_id",
            "foreignField": "_id",
            "as": "relation"
        }
    },
    {"$unwind": "$relation"},

    # optional: remove self-loop
    {
        "$match": {
            "$expr": {"$ne": ["$subject_id", "$object_id"]}
        }
    },

    # final projection
    {
        "$project": {
            "_id": 0,
            "subject": "$subject.name",
            "relation": "$relation.name",
            "object": "$object.name",
            "documents": 1
        }
    }
]

In [28]:
triplets = list(db.triplets.aggregate(pipeline))

print(f"Extracted Triplets ({len(triplets)}):")
for t in triplets:
    print(t)

Extracted Triplets (62):
{'documents': [{'section_id': '4a442face4bb0b158191244aefa6907a33f83f12c2bd7e573a5d4b20e5fc6c5a', 'so_hieu': '90/2025/QH15'}], 'subject': 'pháp luật', 'relation': 'có', 'object': 'yêu cầu'}
{'documents': [{'section_id': 'b05e7c3a4ba876254b35615f2246ecaa2ff4f466c250474133fef6e20623d4b4', 'so_hieu': '57/2024/QH15'}], 'subject': 'pháp luật', 'relation': 'đầu tư', 'object': 'thẩm quyền'}
{'documents': [{'section_id': '94c26dcfb6d7f07e0b0ff27870bf7312f344266aa21385bfe88b648bb0a3ecc5', 'so_hieu': '57/2024/QH15'}], 'subject': 'pháp luật', 'relation': 'phù hợp', 'object': 'quy định'}
{'documents': [{'section_id': 'e34bc6dbf74c0fc2171b8e6e5c5b649c7e51b244a8ee9c5d6de87597d4778ec7', 'so_hieu': '31/2024/QH15'}, {'section_id': '3989125826d6fa8c6f053b49eedc64180aba43c4e4cbeca28c557c49c0eb4e11', 'so_hieu': '258/2025/NĐ-CP'}, {'section_id': '090fe9a592e1a4ac89bb70ac11ff1ebfa018a8508d53fed0177e3e813e435423', 'so_hieu': '94/2024/NĐ-CP'}, {'section_id': 'e2c39b49645b388af7244314b

In [29]:
from bson import ObjectId

# 1. Collect section_ids from triplets
section_ids = set()

for t in triplets:
    for doc in t.get("documents", []):
        section_ids.add(doc["section_id"])

section_ids = list(section_ids)

# 2. Fetch direct legal sections
legal_sections = list(
    db.legal_sections.find(
        {"_id": {"$in": section_ids}},
        {
            "_id": 1,
            "title": 1,
            "content": 1,
            "full_path": 1,
            "so_hieu": 1,
            "document_title": 1
        }
    )
)

section_map = {s["_id"]: s for s in legal_sections}

# 3. Fetch relations involving these sections
relations = list(
    db.legal_section_relations.find(
        {
            "$or": [
                {"source": {"$in": section_ids}},
                {"target": {"$in": section_ids}}
            ]
        }
    )
)

# 4. Collect related section ids
related_section_ids = set()

for r in relations:
    if r["source"] in section_ids:
        related_section_ids.add(r["target"])
    if r["target"] in section_ids:
        related_section_ids.add(r["source"])

# Remove already fetched sections
related_section_ids -= set(section_ids)

related_section_ids = list(related_section_ids)

# 5. Fetch related legal sections
related_sections = list(
    db.legal_sections.find(
        {"_id": {"$in": related_section_ids}},
        {
            "_id": 1,
            "title": 1,
            "content": 1,
            "full_path": 1,
            "so_hieu": 1,
            "document_title": 1
        }
    )
)

related_section_map = {s["_id"]: s for s in related_sections}

# 6. Enrich triplets
for t in triplets:
    enriched_sections = []
    related_enriched_sections = []

    for doc in t.get("documents", []):
        sid = doc["section_id"]

        if sid in section_map:
            enriched_sections.append(section_map[sid])

        # Attach related sections via relations
        for r in relations:
            if r["source"] == sid and r["target"] in related_section_map:
                related_enriched_sections.append({
                    **related_section_map[r["target"]],
                    "relation_type": r["type"],
                    "amendment_types": r.get("amendment_types", [])
                })

            if r["target"] == sid and r["source"] in related_section_map:
                related_enriched_sections.append({
                    **related_section_map[r["source"]],
                    "relation_type": r["type"],
                    "amendment_types": r.get("amendment_types", [])
                })

    t["legal_sections"] = enriched_sections
    t["related_legal_sections"] = related_enriched_sections

# 7. Print result
for i, t in enumerate(triplets, 1):
    print(f"\nTriplet {i}")
    print("Subject :", t["subject"])
    print("Relation:", t["relation"])
    print("Object  :", t["object"])

    for s in t.get("legal_sections", []):
        print("  └─ Văn bản:", s["document_title"])
        print("     Số hiệu:", s["so_hieu"])
        print("     Vị trí :", s["full_path"])
        print("     Tiêu đề:", s["title"])
        print("     Nội dung:", s["content"])

    for s in t.get("related_legal_sections", []):
        print("  └─ (Liên quan)")
        print("     Văn bản:", s["document_title"])
        print("     Số hiệu:", s["so_hieu"])
        print("     Vị trí :", s["full_path"])
        print("     Quan hệ:", s["relation_type"])
        print("     Kiểu sửa:", s["amendment_types"])

print(f"Total Sections Retrieved: {len(legal_sections) + len(related_sections)}")


Triplet 1
Subject : pháp luật
Relation: có
Object  : yêu cầu
  └─ Văn bản: Luật sửa đổi, bổ sung một số điều của luật đấu thầu, luật đầu tư theo phương thức đối tác công tư, luật hải quan, luật thuế giá trị gia tăng, luật thuế xuất khẩu, thuế nhập khẩu, luật đầu tư, luật đầu tư công, luật quản lý, sử dụng tài sản công
     Số hiệu: 90/2025/QH15
     Vị trí : 90/2025/QH15_điều 10_khoản 4_điểm a
     Tiêu đề: điểm a
     Nội dung: Kể từ ngày Luật này có hiệu lực thi hành, hồ sơ hợp lệ đề nghị chấp thuận, điều chỉnh chủ trương đầu tư dự án đầu tư có yêu cầu di dân tái định cư từ 10.000 người trở lên ở miền núi, từ 20.000 người trở lên ở vùng khác; dự án đầu tư xây dựng mới: cảng hàng không, sân bay; đường cất hạ cánh của cảng hàng không, sân bay; nhà ga hành khách của cảng hàng không quốc tế; nhà ga hàng hóa của cảng hàng không, sân bay có công suất từ 01 triệu tấn/năm trở lên; dự án đầu tư mới kinh doanh vận chuyển hành khách bằng đường hàng không; dự án đầu tư xây dựng mới: bến cảng, k

In [30]:
print(question)

Luật sư cho hỏi; gđ e nộp đơn hoà giải tranh chấp đất đến nay đã hơn 50 ngày mà ubnd xã họ cứ hẹn qua tuần mà 2 tuần nay vẫn chưa giãi quyết. Chỗ phần đất đang tranh chấp thì họ đang sử dụng bt. Vậy gđ e phải làm ntn ạ e cám ơn


## Reranking

In [31]:
import faiss
import pickle

sections_index = faiss.read_index("sections.faiss")

with open("section_id_map.pkl", "rb") as f:
    vector_to_section = pickle.load(f)

In [71]:
import faiss
import numpy as np
from rank_bm25 import BM25Okapi
from typing import List, Dict

TOP_K = 10

def get_openai_embedding(text: str, model: str = "text-embedding-3-large") -> np.ndarray:
    response = gpt_client.embeddings.create(
        input=text,
        model=model
    )
    emb = np.array(response.data[0].embedding, dtype="float32")
    return emb / np.linalg.norm(emb)

def preprocess_text(text: str):
    return text.lower().split()

section_to_vector = {sid: idx for idx, sid in enumerate(vector_to_section)}

def get_section_embedding(section_id: str):
    idx = section_to_vector.get(section_id)
    if idx is None:
        return None
    return sections_index.reconstruct(idx)

def rerank_sections(
    query: str,
    sections: List[Dict],
    query_emb: np.ndarray,
):
    if not sections:
        return []

    texts = [
        f"{s.get('title', '')} {s.get('content', '')} {s.get('document_title', '')}"
        for s in sections
    ]

    tokenized_corpus = [preprocess_text(t) for t in texts]
    bm25 = BM25Okapi(tokenized_corpus)
    bm25_scores = bm25.get_scores(preprocess_text(query))
    if bm25_scores.max() > 0:
        bm25_scores = bm25_scores / bm25_scores.max()

    dpr_scores = []
    for s in sections:
        emb = get_section_embedding(s["_id"])
        if emb is None:
            dpr_scores.append(0.0)
        else:
            dpr_scores.append(float(np.dot(query_emb, emb)))

    dpr_scores = np.array(dpr_scores)
    if dpr_scores.max() > 0:
        dpr_scores = dpr_scores / dpr_scores.max()

    combined_scores = 0.3 * bm25_scores + 0.7 * dpr_scores
    ranked_idx = np.argsort(combined_scores)[::-1]

    results = []
    for i in ranked_idx:
        sec = sections[i].copy()
        sec["bm25_score"] = float(bm25_scores[i])
        sec["dpr_score"] = float(dpr_scores[i])
        sec["combined_score"] = float(combined_scores[i])
        results.append(sec)

    return results

In [72]:
query_emb = get_openai_embedding(question)

for t in triplets:
    triplet_section_ids = {d["section_id"] for d in t.get("documents", [])}

    direct_candidates = [
        section_map[sid]
        for sid in triplet_section_ids
        if sid in section_map
    ]

    related_candidates = []
    for r in relations:
        if r["source"] in triplet_section_ids and r["target"] in related_section_map:
            related_candidates.append({
                **related_section_map[r["target"]],
                "relation_type": r["type"],
                "amendment_types": r.get("amendment_types", [])
            })
        if r["target"] in triplet_section_ids and r["source"] in related_section_map:
            related_candidates.append({
                **related_section_map[r["source"]],
                "relation_type": r["type"],
                "amendment_types": r.get("amendment_types", [])
            })

    t["legal_sections"] = rerank_sections(
        question,
        direct_candidates,
        query_emb,
    )

    t["related_legal_sections"] = rerank_sections(
        question,
        related_candidates,
        query_emb,
    )

In [79]:
from src.retrieval.utils.collect_content import collect_sections_content_upward

TOP_K_RERANKING = 20

all_ranked = {}

for t in triplets:
    for s in t.get("legal_sections", []):
        sid = s["_id"]
        score = s["combined_score"]
        if sid not in all_ranked or score > all_ranked[sid]["combined_score"]:
            all_ranked[sid] = s

    for s in t.get("related_legal_sections", []):
        sid = s["_id"]
        score = s["combined_score"]
        if sid not in all_ranked or score > all_ranked[sid]["combined_score"]:
            all_ranked[sid] = s

top_k_result = sorted(
    all_ranked.values(),
    key=lambda x: x["combined_score"],
    reverse=True
)[:TOP_K_RERANKING]

sections_content = collect_sections_content_upward(db.legal_sections, [s['_id'] for s in top_k_result])
for section in top_k_result:
    section['content'] = sections_content.get(section['_id'], section.get('content', ''))

for i, s in enumerate(top_k_result, 1):
    print(f"{i}. {s.get('full_path', '')}")
    print(f"   section_id: {s['_id']}")
    print(f"   so_hieu  : {s.get('so_hieu', '')}")
    print(f"   combined_score: {s['combined_score']:.4f}")
    print(f"   content  :\n {s.get('content', '')}")
    print()

1. 31/2024/QH15_chương xi_mục 2_điều 162_khoản 1_điểm c
   section_id: e34bc6dbf74c0fc2171b8e6e5c5b649c7e51b244a8ee9c5d6de87597d4778ec7
   so_hieu  : 31/2024/QH15
   combined_score: 1.0000
   content  :
 quyền và nghĩa vụ của tổ chức tư vấn xác định giá đất
Tổ chức tư vấn xác định giá đất có các quyền sau đây:
Quyền khác theo quy định của pháp luật.

2. 27/2023/QH15_chương xiii_điều 198_khoản 5_điểm đ
   section_id: 0eb5c45cdb7f1e757086ecb94ea939aa95d9178208ad7fcea8c43a47f262672c
   so_hieu  : 27/2023/QH15
   combined_score: 1.0000
   content  :
 quy định chuyển tiếp
Quy định chuyển tiếp đối với quy định tại Chương VI của Luật này như sau:
Đối với trường hợp bán nhà ở xã hội phải nộp tiền sử dụng đất theo quy định của pháp luật về nhà ở trước ngày Luật này có hiệu lực thi hành mà đến ngày Luật này có hiệu lực thi hành vẫn chưa nộp tiền sử dụng đất thì tiếp tục nộp tiền theo quy định của pháp luật về nhà ở trước ngày Luật này có hiệu lực thi hành;

3. 140/2025/NĐ-CP_chương vi_điều 32_kh

## Answer Generation

In [81]:
all_so_hieu = set()
for s in top_k_result:
    so_hieu = s.get("so_hieu")
    if so_hieu:
        all_so_hieu.add(so_hieu)

documents = db.documents.find({"so_hieu": {"$in": list(all_so_hieu)}})
documents_by_so_hieu = {
    doc["so_hieu"]: doc
    for doc in documents
}

In [82]:
import json

llm_payload = []

for i, s in enumerate(top_k_result, 1):
    so_hieu = s.get("so_hieu")
    document = documents_by_so_hieu.get(so_hieu, {}) if so_hieu else {}
    effective_date_str = document["effective_date"].strftime("%Y-%m-%d") if document.get("effective_date") else ""
    doc_title = document.get("title", "").strip().replace("\n", " ")

    llm_payload.append({
        "so_hieu": s.get("so_hieu", ""),
        "document_title": doc_title,
        "full_path": s.get("full_path", "").replace("_", ", "),
        "effective_date": effective_date_str,
        "content": s.get("content", "")
    })

print(question)
context_str = json.dumps(llm_payload, ensure_ascii=False, indent=2)
print(context_str)

Luật sư cho hỏi; gđ e nộp đơn hoà giải tranh chấp đất đến nay đã hơn 50 ngày mà ubnd xã họ cứ hẹn qua tuần mà 2 tuần nay vẫn chưa giãi quyết. Chỗ phần đất đang tranh chấp thì họ đang sử dụng bt. Vậy gđ e phải làm ntn ạ e cám ơn
[
  {
    "so_hieu": "31/2024/QH15",
    "document_title": "Luật Đất đai",
    "full_path": "31/2024/QH15, chương xi, mục 2, điều 162, khoản 1, điểm c",
    "effective_date": "2024-01-18",
    "content": "quyền và nghĩa vụ của tổ chức tư vấn xác định giá đất\nTổ chức tư vấn xác định giá đất có các quyền sau đây:\nQuyền khác theo quy định của pháp luật."
  },
  {
    "so_hieu": "27/2023/QH15",
    "document_title": "LUẬT NHÀ Ở",
    "full_path": "27/2023/QH15, chương xiii, điều 198, khoản 5, điểm đ",
    "effective_date": "2023-11-27",
    "content": "quy định chuyển tiếp\nQuy định chuyển tiếp đối với quy định tại Chương VI của Luật này như sau:\nĐối với trường hợp bán nhà ở xã hội phải nộp tiền sử dụng đất theo quy định của pháp luật về nhà ở trước ngày Luật này

In [83]:
response_qa = gpt_client.responses.create(
    model="gpt-5.1",
    input=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT_QA
        },
        {
            "role": "user",
            "content": f"Ngữ cảnh: {context_str}.\n\n Câu hỏi: {question}. \n\n Hãy trả lời câu hỏi dựa hoàn toàn trên ngữ cảnh đã cho."
        }
    ]
)

answer = response_qa.output_text
print(answer)

{
  "answer": "Trong các văn bản được cung cấp không có quy định nào về:\n- Thời hạn Ủy ban nhân dân cấp xã phải tổ chức hòa giải tranh chấp đất đai;\n- Trình tự, thủ tục hòa giải tranh chấp đất đai tại UBND cấp xã;\n- Quyền của gia đình bạn trong trường hợp UBND xã chậm tổ chức hòa giải.\n\nCác quy định được trích dẫn trong ngữ cảnh chủ yếu liên quan đến:\n- Thẩm quyền xử phạt vi phạm hành chính về đất đai của Chủ tịch UBND cấp xã (phạt tiền, tịch thu giấy tờ giả, buộc khôi phục lại tình trạng ban đầu của đất, mốc địa giới) [Nghị định 123/2024/NĐ-CP, chương III, điều 30, khoản 1, các điểm b, c, d];\n- Quản lý, sử dụng đất thuộc hành lang bảo vệ an toàn công trình, thu hồi đất, bồi thường, hỗ trợ, tái định cư [Nghị định 102/2024/NĐ-CP, chương VII, mục 6, điều 97, khoản 1, điểm b, c];\n- Quyền, nghĩa vụ trong lĩnh vực nhà ở, nhà ở xã hội, nhà ở công vụ và các quy định chuyển tiếp [Luật Nhà ở 27/2023/QH15, nhiều điều, khoản, điểm];\n- Một số quy định về doanh nghiệp, đầu tư công, hợp đồn

## Result

In [9]:
import json
import csv

JSONL_PATH = r"E:\Github\LawAssistant\notebook\test_new_retrival\QA_batch_requests\batch_qa_output.jsonl"
CSV_PATH = r"E:\Github\LawAssistant\notebook\test_new_retrival\QA_batch_requests\batch_qa_output.csv"

rows = []

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)

        if "response" not in r or r["response"]["status_code"] != 200:
            continue

        cus_id = r["custom_id"]
        raw = r["response"]["body"]["choices"][0]["message"]["content"]

        # raw is a JSON string → dict
        content = json.loads(raw)

        rows.append({
            "custom_id": cus_id,
            "answer": content.get("answer", ""),
            # join list into readable text for CSV
            "source": " | ".join(content.get("source", []))
        })

# write CSV
with open(CSV_PATH, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["custom_id", "answer", "source"]
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} rows to CSV")

Saved 120 rows to CSV
